# NB01: Taxonomy baseline

Tests **H1**: CLR-based taxonomy (B1) predicts soil metals better than cheap geochem alone (B2).

Models evaluated (spatial leave-one-block-out CV, k=5 geographic blocks):
- **B0**: Intercept-only (training mean)
- **B1**: CLR-transformed genus RA (XGBoost)
- **B2**: pH + lat + lon (ridge)
- **B3**: CLR + pH + lat + lon (XGBoost)

**H1 success criterion**: B1 RMSE ≤ B2 RMSE for ≥2 of 4 target metals (bootstrap CI excludes 0 in the positive direction for ΔRMSE = B2 − B1).

**Outputs**
- `data/cv_results_baselines.csv` — per-fold RMSE for B0, B1, B2, B3
- `data/bootstrap_h1.csv` — bootstrap ΔRMSE (B2 − B1) CI

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIG_DIR  = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())

from modelling import TARGETS, run_spatial_block_cv, rmse, get_features
from evaluation import plot_rmse_comparison

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
blocks = pd.read_csv(DATA_DIR / 'spatial_blocks.csv', index_col=0)['block']
print(f'Feature matrix: {feature_matrix.shape}')
print(f'Blocks: {blocks.value_counts().to_dict()}')

## 1. B0: Intercept-only baseline

In [ ]:
b0_records = []
for target in TARGETS:
    if target not in feature_matrix.columns:
        continue
    y = feature_matrix[target].dropna()
    b_aligned = blocks.reindex(feature_matrix.index)
    for test_block in b_aligned.dropna().unique():
        train_mask = (b_aligned != test_block) & b_aligned.notna()
        test_mask  = (b_aligned == test_block)
        y_train = y[y.index.isin(feature_matrix[train_mask].index)].dropna()
        y_test  = y[y.index.isin(feature_matrix[test_mask].index)].dropna()
        if len(y_test) < 5:
            continue
        b0_records.append({
            'model': 'B0', 'target': target, 'block': test_block,
            'n_train': len(y_train), 'n_test': len(y_test),
            'rmse': rmse(y_test.values, np.full(len(y_test), y_train.mean())),
        })

b0_df = pd.DataFrame(b0_records)
print('B0 mean RMSE by target:')
print(b0_df.groupby('target')['rmse'].mean().round(4))

## 2. B1 (CLR), B2 (geochem), B3 (CLR+geochem)

In [ ]:
all_results = [b0_df]

for model_name, mtype in [('B1', 'xgboost'), ('B2', 'ridge'), ('B3', 'xgboost')]:
    print(f'Running {model_name}...')
    for target in TARGETS:
        if target not in feature_matrix.columns:
            continue
        res, _ = run_spatial_block_cv(
            feature_matrix, target, model_name, blocks, model_type=mtype,
        )
        all_results.append(res)
    print(f'  Done.')

cv_baselines = pd.concat(all_results, ignore_index=True)
cv_baselines.to_csv(DATA_DIR / 'cv_results_baselines.csv', index=False)

pivot = cv_baselines.groupby(['model', 'target'])['rmse'].mean().unstack('target').round(4)
print('\nMean spatial-CV RMSE:')
print(pivot)

## 3. H1: Bootstrap ΔRMSE (B2 − B1)

**Success**: ΔRMSE CI excludes 0 in the positive direction for ≥2 of 4 metals.

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(42)

h1_records = []
for target in TARGETS:
    if target not in feature_matrix.columns:
        continue
    y_all = feature_matrix[target].dropna()
    b1_preds = pd.Series(np.nan, index=feature_matrix.index)
    b2_preds = pd.Series(np.nan, index=feature_matrix.index)

    b_aligned = blocks.reindex(feature_matrix.index)
    for test_block in b_aligned.dropna().unique():
        test_mask  = b_aligned == test_block
        train_mask = ~test_mask & b_aligned.notna()

        from modelling import _drop_nan_rows, build_xgboost, build_ridge

        X1 = get_features(feature_matrix, 'B1')
        X2 = get_features(feature_matrix, 'B2')
        y  = feature_matrix[target]

        Xt1, yt1 = _drop_nan_rows(X1[train_mask], y[train_mask])
        Xs1, ys1 = _drop_nan_rows(X1[test_mask],  y[test_mask])
        Xt2, yt2 = _drop_nan_rows(X2[train_mask], y[train_mask])
        Xs2, ys2 = _drop_nan_rows(X2[test_mask],  y[test_mask])

        if len(Xs1) < 5 or len(Xs2) < 5:
            continue

        m1 = build_xgboost(); m1.fit(Xt1, yt1); b1_preds.loc[Xs1.index] = m1.predict(Xs1)
        m2 = build_ridge();   m2.fit(Xt2, yt2); b2_preds.loc[Xs2.index] = m2.predict(Xs2)

    valid = y_all.index[b1_preds[y_all.index].notna() & b2_preds[y_all.index].notna()]
    y_v  = y_all.loc[valid].values
    p1_v = b1_preds.loc[valid].values
    p2_v = b2_preds.loc[valid].values

    obs_delta = rmse(y_v, p2_v) - rmse(y_v, p1_v)
    boot_deltas = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(y_v), len(y_v))
        boot_deltas.append(rmse(y_v[idx], p2_v[idx]) - rmse(y_v[idx], p1_v[idx]))
    lo, hi = np.percentile(boot_deltas, [2.5, 97.5])
    h1_records.append({
        'target': target,
        'delta_rmse_b2_minus_b1': obs_delta,
        'ci_lo': lo, 'ci_hi': hi,
        'h1_pass': lo > 0,
    })

h1_df = pd.DataFrame(h1_records)
h1_df.to_csv(DATA_DIR / 'bootstrap_h1.csv', index=False)
print('H1 results (B2 − B1 ΔRMSE):')
print(h1_df.to_string(index=False))
n_pass = h1_df['h1_pass'].sum()
print(f'\nH1 OUTCOME: {"SUPPORTED" if n_pass >= 2 else "NOT SUPPORTED"} ({n_pass}/4 metals pass)')

## 4. RMSE comparison plot

In [ ]:
plot_rmse_comparison(
    cv_baselines,
    models=['B0', 'B1', 'B2', 'B3'],
    targets=[t for t in TARGETS if t in feature_matrix.columns],
    out_path=FIG_DIR / 'baseline_rmse_comparison.png',
)